# Disaggregated inference comparisons

Load measured CSVs from `tracks/nvidia-dynamo/studies/`. No example performance numbers are included.

Compare matching workloads and budgets, inspect errors first, and retain dataset/model/image identifiers. ITL is shown only where explicit one-token events match server usage. TPOT and streaming chunk intervals remain separate. This notebook evaluates serving performance, not agent task success.

In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'benchmarks').is_dir())
sys.path.insert(0, str(ROOT))
from benchmarks.collect import collect
RESULTS = Path(os.environ.get('DISAGG_RESULTS', str(ROOT / 'results')))
EXPORT = RESULTS / 'analysis'
summary_file = collect(RESULTS)
runs = pd.read_csv(summary_file) if summary_file else pd.DataFrame()
if not runs.empty:
    # Historical files from this repository predate backend selection and use vLLM.
    if 'backend' not in runs:
        runs['backend'] = 'vllm'
    runs['backend'] = runs['backend'].fillna('vllm')
print(f'{len(runs)} measured runs')
plt.style.use('seaborn-v0_8-whitegrid')

## Check failures and actual lengths

Invalid samples remain visible here and are excluded from comparison charts. A successful response below the requested long-context minimum is invalid. Inspect `requests.csv`/`requests.jsonl` for individual failures. Context-window configuration does not establish actual prompt length.

In [ ]:
audit_columns = ['run_id', 'model', 'technology', 'backend', 'workload', 'concurrency',
                 'requests', 'failed_requests', 'invalid_measurements',
                 'prompt_tokens_mean', 'completion_tokens_mean', 'itl_coverage',
                 'model_revision', 'image', 'dataset_sha256', 'extra_body_json']
if not runs.empty:
    display(runs[[c for c in audit_columns if c in runs]])
    valid = runs[(runs.failed_requests == 0) & (runs.invalid_measurements == 0)
                 & (runs.valid_measurements > 0)].copy()
    print(f'{len(valid)}/{len(runs)} fully valid runs available for comparison')
else:
    valid = pd.DataFrame()
    print('Run benchmarks.run and benchmarks.collect, then rerun this notebook.')

## Concurrency sweep

Use this view first when tuning one deployment. Each line keeps the model, dataset,
deployment, output budget, cache label and request settings fixed while concurrency
changes. Points are medians of repeated runs; p95 values are medians of per-run
p95s, not pooled percentiles. The exported CSV records the full series identifiers.
Check actual output lengths and failures above before interpreting throughput.

In [ ]:
sweep = pd.DataFrame()
if not valid.empty:
    sweep_keys = ['model', 'technology', 'backend', 'dataset_sha256', 'deployment_sha256',
                  'extra_body_json', 'image', 'model_revision', 'workload', 'max_model_len',
                  'requested_output_tokens', 'min_input_tokens', 'cache_state', 'temperature',
                  'session_rate', 'think_time_s', 'sessions', 'gpu_count']
    sweep_keys = [key for key in sweep_keys if key in valid.columns]
    sweep = valid.groupby(sweep_keys + ['concurrency'], dropna=False).agg(
        repeats=('run_id', 'count'),
        output_tps=('output_throughput_tps', 'median'),
        output_tps_min=('output_throughput_tps', 'min'),
        output_tps_max=('output_throughput_tps', 'max'),
        ttft_p95_ms=('ttft_ms_p95', 'median'),
        tpot_p95_ms=('tpot_ms_p95', 'median'),
        mean_output_tokens=('completion_tokens_mean', 'mean'),
    ).reset_index()
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), layout='constrained')
    for series_id, (_, frame) in enumerate(sweep.groupby(sweep_keys, dropna=False)):
        frame = frame.sort_values('concurrency')
        sweep.loc[frame.index, 'series_id'] = series_id
        first = frame.iloc[0]
        label = f"{first['model'].split('/')[-1]} / {first['technology']} / {first['backend']} [{series_id}]"
        for ax, metric, title in zip(axes, ['output_tps', 'ttft_p95_ms', 'tpot_p95_ms'],
                                    ['Output tokens/s', 'p95 TTFT, ms', 'p95 TPOT, ms']):
            ax.plot(frame.concurrency, frame[metric], marker='o', label=label)
            ax.set_xlabel('Concurrent sessions')
            ax.set_title(title)
            ax.set_ylim(bottom=0)
    axes[0].legend(fontsize=7)
    display(sweep)
    EXPORT.mkdir(parents=True, exist_ok=True)
    sweep.to_csv(EXPORT / 'concurrency-sweep.csv', index=False)
    fig.savefig(EXPORT / 'concurrency-sweep.png', dpi=180, bbox_inches='tight')
    plt.show()


## Select one comparable workload

Choose `COHORT_INDEX` after inspecting the cohort table. Models, technologies and backends vary inside a cohort; workload, concurrency, token budgets, cache label, load mode, temperature and think time are fixed. Dataset hashes and reasoning settings remain visible. Equal-text cross-model tests and equal-token-budget tests answer different questions; state your choice when exporting results.

In [ ]:
cohort_keys = ['workload', 'concurrency', 'max_model_len', 'requested_output_tokens',
               'min_input_tokens', 'cache_state', 'temperature', 'session_rate', 'think_time_s']
COHORT_INDEX = 0
cohort = pd.DataFrame()
if not valid.empty:
    cohorts = valid[cohort_keys].drop_duplicates().reset_index(drop=True)
    display(cohorts)
    chosen = cohorts.iloc[COHORT_INDEX]
    mask = (valid[cohort_keys] == chosen).all(axis=1)
    cohort = valid.loc[mask].copy()
    display(cohort[['run_id', 'model', 'technology', 'backend', 'dataset_sha256', 'extra_body_json',
                    'image', 'model_revision', 'deployment_sha256']])

## Group identical repetitions

Each row summarizes repetitions with the same model, technology, backend, dataset, deployment and request options. Rates are medians of independent runs. Report repeat counts and spread; do not average per-run p99s and call the result a pooled p99. The displayed tail figures below are explicitly medians of per-run tails.

In [ ]:
group_keys = ['model', 'technology', 'backend', 'dataset_sha256', 'deployment_sha256',
              'extra_body_json', 'image', 'model_revision']
comparison = pd.DataFrame()
if not cohort.empty:
    comparison = cohort.groupby(group_keys, dropna=False).agg(
        repeats=('run_id', 'count'),
        output_tps=('output_throughput_tps', 'median'),
        output_tps_std=('output_throughput_tps', 'std'),
        input_tps=('input_throughput_tps', 'median'),
        ttft_p50_ms=('ttft_ms_p50', 'median'),
        ttft_p99_ms=('ttft_ms_p99', 'median'),
        tpot_p50_ms=('tpot_ms_p50', 'median'),
        tpot_p99_ms=('tpot_ms_p99', 'median'),
        itl_p99_ms=('itl_ms_p99', 'median'),
        itl_coverage=('itl_coverage', 'mean'),
        mean_input_tokens=('prompt_tokens_mean', 'mean'),
        mean_output_tokens=('completion_tokens_mean', 'mean'),
    ).reset_index()
    comparison['output_tps_per_gpu'] = comparison.output_tps / 16
    display(comparison)
    if not comparison.duplicated(['model', 'technology', 'backend']).any():
        display(comparison.pivot(index='model', columns=['technology', 'backend'],
                                 values=['output_tps', 'ttft_p50_ms', 'tpot_p50_ms']))
    else:
        print('Multiple dataset/deployment/settings variants for a model+technology+backend: inspect rows separately before pivoting.')
    EXPORT.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(EXPORT / f'comparison-cohort-{COHORT_INDEX}.csv', index=False)

## Throughput and latency side by side

Bars preserve distinct configuration rows. TPS/GPU divides by the fixed 16-GPU allocation, including both P/D pools. Missing ITL values mean unavailable token-resolved timing, not zero latency. Output TPS also depends on actual output lengths and reasoning behavior.

In [ ]:
if not comparison.empty:
    labels = [f"{r.model.split('/')[-1]}\n{r.technology} / {r.backend} [{i}]" for i, r in comparison.iterrows()]
    fig, axes = plt.subplots(1, 3, figsize=(19, max(4, len(labels) * .6)), layout='constrained')
    for ax, column, title in zip(axes, ['output_tps', 'ttft_p50_ms', 'tpot_p50_ms'],
                                 ['Output tokens/s (higher is better)',
                                  'Median TTFT, ms (lower is better)',
                                  'Median TPOT, ms (lower is better)']):
        ax.barh(np.arange(len(labels)), comparison[column], color='#267e96')
        ax.set_yticks(np.arange(len(labels)), labels=labels if ax is axes[0] else [])
        ax.set_title(title)
        ax.set_xlim(left=0)
        ax.invert_yaxis()
    fig.suptitle(f"{chosen['workload']} | concurrency {chosen['concurrency']} | context limit {chosen['max_model_len']:,}")
    fig.savefig(EXPORT / f'comparison-cohort-{COHORT_INDEX}.png', dpi=180, bbox_inches='tight')
    plt.show()

## Request-level latency distributions

These ECDFs pool requests from the selected repeated runs and keep distinct deployment/dataset variants separate. TTFT includes network and parser buffering. Responses with more than one token per event cannot reveal individual ITLs.

In [ ]:
requests = pd.DataFrame()
if not cohort.empty:
    frames = []
    for row in cohort.itertuples():
        path = RESULTS / row.run_id / 'requests.csv'
        if path.exists():
            frame = pd.read_csv(path)
            frame['series'] = (row.model.split('/')[-1] + ' / ' + row.technology + ' / ' + row.backend +
                               ' / ' + row.deployment_sha256[:6] + ' / ' + row.dataset_sha256[:6] +
                               ' / ' + row.extra_body_json)
            frames.append(frame)
    if frames:
        requests = pd.concat(frames, ignore_index=True)
        requests = requests[(requests.success == True) & (requests.measurement_valid == True)]
if not requests.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), layout='constrained')
    for name, frame in requests.groupby('series'):
        for ax, metric in zip(axes, ['ttft_ms', 'e2e_ms']):
            values = np.sort(frame[metric].dropna().to_numpy()) / 1000
            if len(values):
                ax.step(values, np.arange(1, len(values) + 1) / len(values), where='post', label=name)
            ax.set_xlabel(metric.replace('_ms', '') + ' (seconds)')
            ax.set_ylabel('Fraction of requests')
    axes[0].legend(fontsize=7, loc='best')
    fig.savefig(EXPORT / f'latency-ecdf-cohort-{COHORT_INDEX}.png', dpi=180, bbox_inches='tight')
    plt.show()

## Interpretation checklist

- Confirm P/D KV transfer and RDMA using both workers' logs/counter deltas.
- Check actual token lengths, errors, cache hits and ITL coverage before comparing rates.
- Use the same dataset hash for technology comparisons. Record tokenizer/count differences for cross-model comparisons.
- Keep non-thinking and thinking runs distinct; Kimi K3 always thinks.
- Cold/warm is a user-supplied label; the runner does not flush caches.
- Different backend versions/images make this a stack comparison, not an isolated comparison of routing overhead.
- Eight example sessions are useful for smoke tests; increase samples and repetitions for tail-latency claims.

Exports are created under `tracks/nvidia-dynamo/studies/analysis/` only when valid runs exist.